# ZeroController: train the action classifier on Google Colab

This notebook trains the low-latency classifier that `run_model.py` uses, on Google's
computers instead of your laptop. It clones the project from GitHub, rebuilds the updated
clips (only the frames that show each action), trains, and gives you
`action_classifier_fast.pth` to copy into `models/` on your laptop.

**How to run:** open this notebook in Colab, check the settings in the first code cell,
then choose **Runtime > Run all**. A GPU runtime (Runtime > Change runtime type) speeds
up the 1-second models. The fast model trains in a few minutes on a CPU too.

If a cell installs packages and Colab asks you to restart the runtime, restart it and
choose Run all again.

In [ ]:
# Settings
REPO = "https://github.com/AbdulRahmanAzam/ZeroController.git"
BRANCH = "fix/camera-accuracy"   # the branch that has train_fast.py, tools/ and data/labels/
USE_DRIVE = True                 # keep trained models (and any rebuilt data) in Google Drive
DRIVE_DIR = "/content/drive/MyDrive/zeroController"
EPOCHS = 12
EVALUATE = False                 # first score the method on each held-out recording session (15-30 min)
WINDOW_MODELS = []               # e.g. ["stgcn", "tcn"]: also train the 1-second models

In [ ]:
import importlib.util
import json
import os
import shutil
import subprocess
import sys


def run(*args):
    """Run a command, show its output here, and stop the notebook if it fails."""
    args = [str(a) for a in args]
    print("$", " ".join(args), flush=True)
    process = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait():
        raise RuntimeError(f"Command failed with exit code {process.returncode}: {' '.join(args)}")


if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)

## 1. Get the code and data
A shallow clone is enough for training: `data/raw/` and `data/labels/` are in the repository.

In [ ]:
ROOT = "/content/ZeroController"
if os.path.isdir(os.path.join(ROOT, ".git")):
    run("git", "-C", ROOT, "fetch", "--depth", "1", "origin", BRANCH)
    run("git", "-C", ROOT, "checkout", "-B", BRANCH, "FETCH_HEAD")
else:
    run("git", "clone", "--depth", "1", "--branch", BRANCH, REPO, ROOT)
os.chdir(ROOT)
sys.path.insert(0, ROOT)
run("git", "log", "-1", "--format=%h %s")

needed = []
if EVALUATE:  # scoring replays clips through run_model.py's gate, which imports these
    needed = [pkg for pkg, module in [("mediapipe", "mediapipe"), ("fastapi", "fastapi"), ("uvicorn[standard]", "uvicorn")]
              if importlib.util.find_spec(module) is None]
if needed:
    run(sys.executable, "-m", "pip", "install", "-q", *needed)

import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

## 2. Camera-condition data (`data/multicam/`)
The same clips re-tracked under live camera conditions (lite model, 4:3, zoom, far, tilt).
Training uses them so the model copes with the camera you play with. The notebook uses,
in order: the copy in the repository, the copy in Google Drive, or a rebuild from the
recorded videos in git history (about 2.7 GB to download and 2-3 hours on a free Colab
CPU, once; finished parts are kept in Drive, so a disconnect does not lose them).

In [ ]:
CONDITIONS = ["full_169", "lite_169", "lite_43", "lite_43_zoom", "lite_43_far", "lite_43_tilt", "full_43"]
CACHE = os.path.join(DRIVE_DIR, "multicam")


def complete(folder):
    return all(os.path.exists(os.path.join(folder, name + ".npz")) for name in CONDITIONS)


if complete("data/multicam"):
    print("data/multicam came with the repository.")
elif USE_DRIVE and complete(CACHE):
    shutil.copytree(CACHE, "data/multicam", dirs_exist_ok=True)
    print("Copied data/multicam from Google Drive.")
else:
    print("Rebuilding data/multicam from the recorded videos in git history.")
    shallow = subprocess.run(["git", "rev-parse", "--is-shallow-repository"],
                             capture_output=True, text=True).stdout.strip() == "true"
    run("git", "fetch", *(["--unshallow"] if shallow else []), "origin", "+refs/heads/*:refs/remotes/origin/*")
    if importlib.util.find_spec("mediapipe") is None:
        run(sys.executable, "-m", "pip", "install", "-q", "mediapipe")
    from config import POSE_MODEL_VARIANTS
    from main import ensure_pose_model
    for variant in ("lite", "full"):
        ensure_pose_model(POSE_MODEL_VARIANTS[variant]["path"], POSE_MODEL_VARIANTS[variant]["url"])
    out = CACHE if USE_DRIVE else "data/multicam"
    os.makedirs(out, exist_ok=True)
    run(sys.executable, "tools/build_multicam_landmarks.py", "--workers", os.cpu_count(), "--out", out)
    if out != "data/multicam":
        shutil.copytree(out, "data/multicam", dirs_exist_ok=True)
print(sorted(os.listdir("data/multicam")))

## 3. Updated clips
`tools/label_segments.py` marks every frame of every recording as action, early, idle or
ignore, and drops recordings with no usable action (hand corrections come from
`data/labels/overrides.csv`). `tools/build_clips.py` then cuts the recordings down to
`data/clips/`: only the frames that show each action, standing frames as idle clips.
Both models train on `data/clips/`.

In [ ]:
run(sys.executable, "tools/label_segments.py")
run(sys.executable, "tools/build_clips.py")

## 4. Optional: score on held-out sessions
With `EVALUATE = True`, trains one model per recording session with that session left out,
then replays the left-out clips through the same gate as `run_model.py`. It reports moves
that fired, delay from the start of each move, wrong triggers, and false triggers per
minute while standing.

In [ ]:
if EVALUATE:
    os.makedirs("/content/holdout", exist_ok=True)
    with open("data/labels/sessions.json") as f:
        sessions = sorted(set(json.load(f).values()))
    for session in sessions:
        run(sys.executable, "train_fast.py", "--holdout-session", session,
            "--save-path", f"/content/holdout/fast_s{session}.pth", "--epochs", EPOCHS)
    for source in ("raw", "lite_43"):
        print(f"\n=== {source} tracking ===")
        run(sys.executable, "tools/eval_fast_stream.py", "--model", "/content/holdout/fast_s{session}.pth",
            "--session", *sessions, "--source", source)
else:
    print("Skipped. Set EVALUATE = True to score the method first.")

## 5. Train the final model on all sessions

In [ ]:
run(sys.executable, "train_fast.py", "--epochs", EPOCHS)
for model_type in WINDOW_MODELS:
    run(sys.executable, "train_robust.py", "--model-type", model_type,
        "--save-path", f"models/action_classifier_{model_type}_robust.pth")

## 6. Save the models

In [ ]:
outputs = ["models/action_classifier_fast.pth"] + [f"models/action_classifier_{m}_robust.pth" for m in WINDOW_MODELS]
if USE_DRIVE:
    target = os.path.join(DRIVE_DIR, "models")
    os.makedirs(target, exist_ok=True)
    for path in outputs:
        shutil.copy(path, target)
    print("Copied to", target)

from google.colab import files
for path in outputs:
    files.download(path)

## Use the new model on your laptop
1. Put `action_classifier_fast.pth` into the `models/` folder of the project, replacing the old one.
2. Run `python run_model.py` as usual.

To train a 1-second model for `run_model.py --classifier window`, rename it to
`models/action_classifier_robust.pth` and set `RUNTIME_MODEL_TYPE` in `config.py` to its type.